In [0]:
# ========================================
# EXPLORATORY NOTEBOOK
# DATASET: gold_dim_product
# ========================================

# ========================================
# 0. CONFIGURATION
# ========================================

CATALOG = "ptfrozenfoods_dev"
SOURCE_SCHEMA = "silver"
TARGET_SCHEMA = "gold"

DOMAIN = "dimensions"
DATASET = "gold_dim_product"

STORAGE_ACCOUNT = "stptfrozenfoodsdevwe01"
SILVER_CONTAINER = "silver"
GOLD_CONTAINER = "gold"

PRODUCTS_DATASET = "erp_products"
ORDER_ITEMS_DATASET = "erp_order_items"

PRODUCTS_TABLE = f"{CATALOG}.{SOURCE_SCHEMA}.{PRODUCTS_DATASET}"
ORDER_ITEMS_TABLE = f"{CATALOG}.{SOURCE_SCHEMA}.{ORDER_ITEMS_DATASET}"

CANDIDATE_TARGET_TABLE = f"{CATALOG}.{TARGET_SCHEMA}.dim_product"

SILVER_BASE_PATH = f"abfss://{SILVER_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"
GOLD_BASE_PATH = f"abfss://{GOLD_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"

PRODUCTS_SOURCE_PATH = f"{SILVER_BASE_PATH}erp/{PRODUCTS_DATASET}/"
ORDER_ITEMS_SOURCE_PATH = f"{SILVER_BASE_PATH}erp/{ORDER_ITEMS_DATASET}/"

CANDIDATE_TARGET_PATH = f"{GOLD_BASE_PATH}dimensions/dim_product/"

In [0]:
# ========================================
# 1. CONTEXT SETUP
# ========================================

print("=" * 80)
print("STARTING GOLD EXPLORATORY: gold_dim_product")
print("=" * 80)

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SOURCE_SCHEMA}")

print("[INFO] Context setup completed successfully.")

STARTING GOLD EXPLORATORY: gold_dim_product
[INFO] Context setup completed successfully.


In [0]:
# ========================================
# 2. CONFIGURATION SUMMARY
# ========================================

print("=" * 80)
print("GOLD EXPLORATORY NOTEBOOK CONFIGURATION")
print("=" * 80)
print(f"Catalog:                         {CATALOG}")
print(f"Source schema:                   {SOURCE_SCHEMA}")
print(f"Target schema:                   {TARGET_SCHEMA}")
print(f"Domain:                          {DOMAIN}")
print(f"Dataset:                         {DATASET}")
print(f"Products table:                  {PRODUCTS_TABLE}")
print(f"Order items table:               {ORDER_ITEMS_TABLE}")
print(f"Candidate target table:          {CANDIDATE_TARGET_TABLE}")
print(f"Products source path:            {PRODUCTS_SOURCE_PATH}")
print(f"Order items source path:         {ORDER_ITEMS_SOURCE_PATH}")
print(f"Candidate target path:           {CANDIDATE_TARGET_PATH}")
print("=" * 80)

GOLD EXPLORATORY NOTEBOOK CONFIGURATION
Catalog:                         ptfrozenfoods_dev
Source schema:                   silver
Target schema:                   gold
Domain:                          dimensions
Dataset:                         gold_dim_product
Products table:                  ptfrozenfoods_dev.silver.erp_products
Order items table:               ptfrozenfoods_dev.silver.erp_order_items
Candidate target table:          ptfrozenfoods_dev.gold.dim_product
Products source path:            abfss://silver@stptfrozenfoodsdevwe01.dfs.core.windows.net/erp/erp_products/
Order items source path:         abfss://silver@stptfrozenfoodsdevwe01.dfs.core.windows.net/erp/erp_order_items/
Candidate target path:           abfss://gold@stptfrozenfoodsdevwe01.dfs.core.windows.net/dimensions/dim_product/


In [0]:
# ========================================
# 3. PRE-CHECKS
# ========================================

print("[INFO] Checking source table availability...")
spark.sql(f"DESCRIBE TABLE {PRODUCTS_TABLE}")
spark.sql(f"DESCRIBE TABLE {ORDER_ITEMS_TABLE}")

print("[INFO] Checking source path access...")
dbutils.fs.ls(PRODUCTS_SOURCE_PATH)
dbutils.fs.ls(ORDER_ITEMS_SOURCE_PATH)

print("[INFO] Checking target container access...")
dbutils.fs.ls(f"abfss://{GOLD_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/")

print("[INFO] Pre-checks completed successfully.")

[INFO] Checking source table availability...
[INFO] Checking source path access...
[INFO] Checking target container access...
[INFO] Pre-checks completed successfully.


In [0]:
# ========================================
# 4. READ SOURCE DATA
# ========================================

print("[INFO] Loading source tables from the Silver layer...")

df_products = spark.table(PRODUCTS_TABLE)
df_order_items = spark.table(ORDER_ITEMS_TABLE)

print("[INFO] Source tables loaded successfully.")

print("=" * 80)
print("SOURCE DATA ROW COUNTS")
print("=" * 80)
print(f"Products:           {df_products.count():,}")
print(f"Order Items:        {df_order_items.count():,}")
print("=" * 80)

# ========================================
# DATASET PREVIEW — INITIAL EXPLORATION
# ========================================

print("=" * 80)
print("DATASET PREVIEW — INITIAL EXPLORATION")
print("=" * 80)
print("[INFO] Displaying sample data from all source datasets...")
print("[INFO] Each preview shows the first 5 records to validate schema and structure.")
print("=" * 80)

# --------------------------------------------------
# PRODUCTS
# --------------------------------------------------
print("\n[INFO] Preview: PRODUCTS (df_products)")
print("-" * 80)
display(df_products.limit(5))

# --------------------------------------------------
# ORDER ITEMS
# --------------------------------------------------
print("\n[INFO] Preview: ORDER ITEMS (df_order_items)")
print("-" * 80)
display(df_order_items.limit(5))

print("\n" + "=" * 80)
print("[INFO] Dataset preview completed successfully.")
print("=" * 80)

[INFO] Loading source tables from the Silver layer...
[INFO] Source tables loaded successfully.
SOURCE DATA ROW COUNTS
Products:           36
Order Items:        245,287
DATASET PREVIEW — INITIAL EXPLORATION
[INFO] Displaying sample data from all source datasets...
[INFO] Each preview shows the first 5 records to validate schema and structure.

[INFO] Preview: PRODUCTS (df_products)
--------------------------------------------------------------------------------


produto_id,produto_nome,categoria,marca,peso_gramas,preco_lista_base,custo_base_unitario,fornecedor_id,data_lancamento,data_fim,status_produto,popularidade_base,sensibilidade_promocao,fator_sazonal_proprio,codigo_barra_legacy,observacao_interna,load_date,ingestion_timestamp,source_file
P036,Atum em óleo 120g,Conservas & Secos,Mar Azul,120,1.1,0.58,F003,2023-09-08,null,Ativo,0.431,0.893,0.902,56000564949,null,2026-03-17,2026-04-03T22:27:58.849Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.core.windows.net/erp/erp_products/load_date=2026-03-17/erp_products.csv
P015,Lasanha bolonhesa 1kg,Pré-cozinhados,Atlântico Select,1000,6.9,4.25,F001,2022-03-03,null,Ativo,2.302,1.291,1.398,56000308650,sku antigo,2026-03-17,2026-04-03T22:27:58.849Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.core.windows.net/erp/erp_products/load_date=2026-03-17/erp_products.csv
P022,Pão de cereais 2kg,Padaria,Doce Norte,2000,4.4,2.1,F005,2022-08-23,null,Ativo,0.715,0.724,0.836,56000153631,sku antigo,2026-03-17,2026-04-03T22:27:58.849Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.core.windows.net/erp/erp_products/load_date=2026-03-17/erp_products.csv
P026,Gelado baunilha 2L 2kg,Sobremesas,PT Frozen Foods,2000,5.9,2.95,F001,2022-04-25,null,Ativo,0.599,1.2,0.7,56000151954,sku antigo,2026-03-17,2026-04-03T22:27:58.849Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.core.windows.net/erp/erp_products/load_date=2026-03-17/erp_products.csv
P021,Pão de água 2400g,Padaria,Padaria Premium,2400,3.9,1.78,F005,2022-10-15,null,Ativo,0.884,1.744,0.789,56000284422,null,2026-03-17,2026-04-03T22:27:58.849Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.core.windows.net/erp/erp_products/load_date=2026-03-17/erp_products.csv



[INFO] Preview: ORDER ITEMS (df_order_items)
--------------------------------------------------------------------------------


item_pedido_id,pedido_id,produto_id,quantidade,preco_lista_unitario,desconto_unitario,preco_venda_unitario,custo_unitario,lote_fornecedor,flag_promocao,nota_item,load_date,ingestion_timestamp,source_file
IT000000027,PED2021011100012,P012,2,5.49,0.0,5.49,3.14,L66962,0,null,2026-03-17,2026-04-03T22:27:43.144Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.core.windows.net/erp/erp_order_items/load_date=2026-03-17/erp_order_items.csv
IT000000061,PED2021011100028,P004,2,8.02,0.0,8.02,5.23,L21140,0,null,2026-03-17,2026-04-03T22:27:43.144Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.core.windows.net/erp/erp_order_items/load_date=2026-03-17/erp_order_items.csv
IT000000087,PED2021011200042,P012,2,5.96,0.48,5.48,3.22,L20459,1,null,2026-03-17,2026-04-03T22:27:43.144Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.core.windows.net/erp/erp_order_items/load_date=2026-03-17/erp_order_items.csv
IT000000946,PED2021012200421,P009,1,2.37,0.05,2.32,1.12,L26896,0,null,2026-03-17,2026-04-03T22:27:43.144Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.core.windows.net/erp/erp_order_items/load_date=2026-03-17/erp_order_items.csv
IT000001177,PED2021012600517,P004,2,7.99,0.0,7.99,4.82,L83053,0,null,2026-03-17,2026-04-03T22:27:43.144Z,abfss://raw@stptfrozenfoodsdevwe01.dfs.core.windows.net/erp/erp_order_items/load_date=2026-03-17/erp_order_items.csv



[INFO] Dataset preview completed successfully.


In [0]:
# ========================================
# 5. SOURCE VALIDATION
# ========================================

print("[INFO] Printing source schemas...")

print("\n[SCHEMA] PRODUCTS")
df_products.printSchema()

print("\n[SCHEMA] ORDER ITEMS")
df_order_items.printSchema()

required_products_columns = [
    "produto_id"
]

required_order_items_columns = [
    "produto_id"
]

missing_products_columns = [c for c in required_products_columns if c not in df_products.columns]
missing_order_items_columns = [c for c in required_order_items_columns if c not in df_order_items.columns]

print(f"[INFO] Missing columns in products: {missing_products_columns}")
print(f"[INFO] Missing columns in order items: {missing_order_items_columns}")

if missing_products_columns:
    raise Exception(f"Missing required columns in products source: {missing_products_columns}")

if missing_order_items_columns:
    raise Exception(f"Missing required columns in order items source: {missing_order_items_columns}")

print("[INFO] Source validation completed successfully.")

[INFO] Printing source schemas...

[SCHEMA] PRODUCTS
root
 |-- produto_id: string (nullable = true)
 |-- produto_nome: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- marca: string (nullable = true)
 |-- peso_gramas: integer (nullable = true)
 |-- preco_lista_base: double (nullable = true)
 |-- custo_base_unitario: double (nullable = true)
 |-- fornecedor_id: string (nullable = true)
 |-- data_lancamento: date (nullable = true)
 |-- data_fim: date (nullable = true)
 |-- status_produto: string (nullable = true)
 |-- popularidade_base: double (nullable = true)
 |-- sensibilidade_promocao: double (nullable = true)
 |-- fator_sazonal_proprio: double (nullable = true)
 |-- codigo_barra_legacy: string (nullable = true)
 |-- observacao_interna: string (nullable = true)
 |-- load_date: date (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)


[SCHEMA] ORDER ITEMS
root
 |-- item_pedido_id: string (nullable = 

In [0]:
# ========================================
# 6. INITIAL DATA QUALITY CHECKS
# ========================================

from pyspark.sql import functions as F

print("[INFO] Checking nulls and distinct keys...")

display(
    df_products.select(
        F.count("*").alias("total_rows"),
        F.countDistinct("produto_id").alias("distinct_produto_id"),
        F.sum(F.when(F.col("produto_id").isNull(), 1).otherwise(0)).alias("null_produto_id")
    )
)

display(
    df_order_items.select(
        F.count("*").alias("total_rows"),
        F.countDistinct("produto_id").alias("distinct_produto_id"),
        F.sum(F.when(F.col("produto_id").isNull(), 1).otherwise(0)).alias("null_produto_id")
    )
)

print("[INFO] Initial data quality checks completed successfully.")

[INFO] Checking nulls and distinct keys...


total_rows,distinct_produto_id,null_produto_id
36,36,0


total_rows,distinct_produto_id,null_produto_id
245287,36,930


[INFO] Initial data quality checks completed successfully.


In [0]:
# ========================================
# 7. PRODUCT KEY COVERAGE CHECK
# ========================================

print("[INFO] Checking product key coverage between products and order items...")

df_products_keys = (
    df_products
    .select("produto_id")
    .dropDuplicates()
)

df_order_items_keys = (
    df_order_items
    .select("produto_id")
    .where(F.col("produto_id").isNotNull())
    .dropDuplicates()
)

products_not_sold = df_products_keys.join(df_order_items_keys, on="produto_id", how="left_anti")
sold_products_not_in_products = df_order_items_keys.join(df_products_keys, on="produto_id", how="left_anti")

print(f"Products master keys:                 {df_products_keys.count():,}")
print(f"Distinct sold product keys:           {df_order_items_keys.count():,}")
print(f"Products not referenced in sales:     {products_not_sold.count():,}")
print(f"Sold products missing in products:    {sold_products_not_in_products.count():,}")

print("\n[INFO] Products not referenced in sales")
display(products_not_sold)

print("\n[INFO] Sold products missing in products")
display(sold_products_not_in_products)

print("[INFO] Product key coverage check completed successfully.")

[INFO] Checking product key coverage between products and order items...
Products master keys:                 36
Distinct sold product keys:           36
Products not referenced in sales:     0
Sold products missing in products:    0

[INFO] Products not referenced in sales


produto_id



[INFO] Sold products missing in products


produto_id


[INFO] Product key coverage check completed successfully.


In [0]:
# ========================================
# 8. DUPLICATE CHECK — PRODUCTS MASTER
# ========================================

print("[INFO] Checking duplicate produto_id in products master...")

df_product_duplicates = (
    df_products
    .groupBy("produto_id")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.col("count").desc(), F.col("produto_id"))
)

duplicate_count = df_product_duplicates.count()

print(f"Duplicate produto_id groups found: {duplicate_count:,}")

display(df_product_duplicates)

print("[INFO] Duplicate check completed successfully.")

[INFO] Checking duplicate produto_id in products master...
Duplicate produto_id groups found: 0


produto_id,count


[INFO] Duplicate check completed successfully.


In [0]:
# ========================================
# 9. ATTRIBUTE QUALITY CHECKS
# ========================================

print("[INFO] Checking nulls in key descriptive attributes...")

display(
    df_products.select(
        F.count("*").alias("total_rows"),
        F.sum(F.when(F.col("produto_id").isNull(), 1).otherwise(0)).alias("null_produto_id"),
        F.sum(F.when(F.col("produto_nome").isNull(), 1).otherwise(0)).alias("null_produto_nome"),
        F.sum(F.when(F.col("categoria").isNull(), 1).otherwise(0)).alias("null_categoria"),
        F.sum(F.when(F.col("marca").isNull(), 1).otherwise(0)).alias("null_marca"),
        F.sum(F.when(F.col("fornecedor_id").isNull(), 1).otherwise(0)).alias("null_fornecedor_id"),
        F.sum(F.when(F.col("status_produto").isNull(), 1).otherwise(0)).alias("null_status_produto"),
        F.sum(F.when(F.col("peso_gramas").isNull(), 1).otherwise(0)).alias("null_peso_gramas")
    )
)

print("[INFO] Attribute quality checks completed successfully.")

[INFO] Checking nulls in key descriptive attributes...


total_rows,null_produto_id,null_produto_nome,null_categoria,null_marca,null_fornecedor_id,null_status_produto,null_peso_gramas
36,0,0,0,0,0,0,0


[INFO] Attribute quality checks completed successfully.


In [0]:
# ========================================
# 10. BUSINESS PROFILE CHECKS
# ========================================

print("[INFO] Profiling business attributes...")

print("\n[INFO] Product status distribution")
display(
    df_products.groupBy("status_produto")
    .count()
    .orderBy(F.col("count").desc())
)

print("\n[INFO] Category distribution")
display(
    df_products.groupBy("categoria")
    .count()
    .orderBy(F.col("count").desc(), F.col("categoria"))
)

print("\n[INFO] Brand distribution")
display(
    df_products.groupBy("marca")
    .count()
    .orderBy(F.col("count").desc(), F.col("marca"))
)

print("[INFO] Business profile checks completed successfully.")

[INFO] Profiling business attributes...

[INFO] Product status distribution


status_produto,count
Ativo,31
Inativo,5



[INFO] Category distribution


categoria,count
Sobremesas,5
Hortícolas,4
Peixe,4
Batatas,3
Bebidas,3
Carne,3
Conservas & Secos,3
Marisco,3
Padaria,3
Pré-cozinhados,3



[INFO] Brand distribution


marca,count
Doce Norte,5
FrioMax,5
PT Frozen Foods,5
Campo Verde,4
Chef Express,4
Atlântico Select,3
Horta Fácil,2
Mar Azul,2
Mesa Pronta,2
Norte Mar,2


[INFO] Business profile checks completed successfully.


In [0]:
# ========================================
# 11. BUILD CANDIDATE DIMENSION
# ========================================

print("[INFO] Building candidate dimension dataset...")

df_dim_product_candidate = (
    df_products
    .select(
        F.col("produto_id"),
        F.col("produto_nome"),
        F.col("categoria"),
        F.col("marca"),
        F.col("peso_gramas"),
        F.col("preco_lista_base"),
        F.col("custo_base_unitario"),
        F.col("fornecedor_id"),
        F.col("data_lancamento"),
        F.col("data_fim"),
        F.col("status_produto"),
        F.col("popularidade_base"),
        F.col("sensibilidade_promocao"),
        F.col("fator_sazonal_proprio")
    )
    .dropDuplicates(["produto_id"])
)

print(f"Candidate dimension row count: {df_dim_product_candidate.count():,}")

display(df_dim_product_candidate.limit(10))

print("[INFO] Candidate dimension dataset created successfully.")

[INFO] Building candidate dimension dataset...
Candidate dimension row count: 36


produto_id,produto_nome,categoria,marca,peso_gramas,preco_lista_base,custo_base_unitario,fornecedor_id,data_lancamento,data_fim,status_produto,popularidade_base,sensibilidade_promocao,fator_sazonal_proprio
P032,"Ice tea limão 1,5L 1500g",Bebidas,Chef Express,1500,1.7,0.86,F006,2021-11-13,2024-03-20,Inativo,4.243,0.929,1.064
P035,Ervilhas em lata 400g,Conservas & Secos,Campo Verde,400,0.9,0.45,F004,2022-09-12,null,Ativo,1.511,0.676,0.782
P012,Hambúrguer bovino 800g,Carne,Chef Express,800,5.9,3.32,F006,2022-04-12,null,Ativo,2.695,0.557,0.922
P016,Empadão de carne 1200g,Pré-cozinhados,Atlântico Select,1200,7.1,4.3,F001,2023-12-06,null,Ativo,2.871,0.982,0.975
P007,Cocktail de marisco 1kg,Marisco,FrioMax,1000,9.8,6.35,F002,2023-09-03,null,Ativo,1.196,1.259,1.155
P031,Sumo de laranja 1L 1kg,Bebidas,Mesa Pronta,1000,1.8,0.92,F006,2023-04-19,null,Ativo,1.202,0.763,1.174
P028,Gelado morango 2L 2kg,Sobremesas,PT Frozen Foods,2000,5.9,2.9,F001,2021-02-07,2023-03-20,Inativo,0.21,1.314,0.89
P030,Tarte de amêndoa 1200g,Sobremesas,Padaria Premium,1200,8.2,4.3,F005,2021-04-27,2022-09-04,Inativo,2.576,0.973,0.833
P003,Potas em anéis 1kg,Peixe,Norte Mar,1000,6.9,3.95,F003,2021-01-14,null,Ativo,0.672,0.938,1.21
P036,Atum em óleo 120g,Conservas & Secos,Mar Azul,120,1.1,0.58,F003,2023-09-08,null,Ativo,0.431,0.893,0.902


[INFO] Candidate dimension dataset created successfully.


In [0]:
# ========================================
# 12. CANDIDATE DIMENSION VALIDATION
# ========================================

print("[INFO] Validating candidate dimension dataset...")

display(
    df_dim_product_candidate.select(
        F.count("*").alias("total_rows"),
        F.countDistinct("produto_id").alias("distinct_produto_id"),
        F.sum(F.when(F.col("produto_id").isNull(), 1).otherwise(0)).alias("null_produto_id")
    )
)

print("[INFO] Candidate dimension validation completed successfully.")

[INFO] Validating candidate dimension dataset...


total_rows,distinct_produto_id,null_produto_id
36,36,0


[INFO] Candidate dimension validation completed successfully.


In [0]:
# ========================================
# 13. MISSING VALUES ANALYSIS — GENERIC
# ========================================

from pyspark.sql import functions as F

def analyze_missing_values(df, dataset_name):
    print("=" * 80)
    print(f"MISSING VALUES ANALYSIS — {dataset_name.upper()}")
    print("=" * 80)

    total_rows = df.count()

    missing_values_df = df.select([
        F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in df.columns
    ])

    missing_values_transposed = (
        missing_values_df
        .select(
            F.array([
                F.struct(
                    F.lit(col_name).alias("column_name"),
                    F.col(col_name).alias("null_count")
                ) for col_name in missing_values_df.columns
            ]).alias("missing_data")
        )
        .select(F.explode("missing_data").alias("data"))
        .select(
            F.col("data.column_name"),
            F.col("data.null_count")
        )
        .withColumn(
            "null_percentage",
            F.round((F.col("null_count") / F.lit(total_rows)) * 100, 4)
        )
        .orderBy(F.col("null_percentage").desc())
    )

    display(missing_values_transposed)

    print(f"[INFO] Total rows analyzed: {total_rows:,}")
    print("[INFO] Missing values analysis completed successfully.")
    print("=" * 80)

analyze_missing_values(df_dim_product_candidate, "dim_product")

MISSING VALUES ANALYSIS — DIM_PRODUCT


column_name,null_count,null_percentage
data_fim,31,86.1111
peso_gramas,0,0.0
custo_base_unitario,0,0.0
data_lancamento,0,0.0
produto_nome,0,0.0
preco_lista_base,0,0.0
categoria,0,0.0
fornecedor_id,0,0.0
produto_id,0,0.0
marca,0,0.0


[INFO] Total rows analyzed: 36
[INFO] Missing values analysis completed successfully.
